In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install transformers datasets pandas tqdm torch

import os

ZIP_PATH = "/content/drive/MyDrive/devgpt-thesis/emotion-labelling/codebert-se-emotion.zip"
MODEL_DIR_CHECK = "/content/drive/MyDrive/devgpt-thesis/emotion-labelling/model/codebert-se-emotion"

if not os.path.exists(MODEL_DIR_CHECK):
    !unzip -q "{ZIP_PATH}" -d "/content/drive/MyDrive/devgpt-thesis/emotion-labelling/model"
    print("Unzipped model to Drive.")
else:
    print("Model already extracted on Drive — skipping unzip.")

if not os.path.exists(MODEL_DIR_CHECK):
    raise FileNotFoundError(f"Model directory not found after unzipping: {MODEL_DIR_CHECK}")
else:
    print(f"Model directory '{MODEL_DIR_CHECK}' successfully created.")

In [ ]:
import pandas as pd
import torch
from transformers import pipeline
from transformers.pipelines.pt_utils import KeyDataset
from datasets import Dataset
from tqdm.auto import tqdm
import os

INPUT_CSV = "/content/drive/MyDrive/devgpt-thesis/emotion-labelling/3_promptscored_dataset_output.csv"
OUTPUT_CSV = "/content/drive/MyDrive/devgpt-thesis/emotion-labelling/3_answerscored_dataset_output.csv"

MODEL_DIR = "/content/drive/MyDrive/devgpt-thesis/emotion-labelling/model/codebert-se-emotion"
BATCH_SIZE = 256

def score_column(classifier, hf_dataset, column_name, desc):
    emotions, scores = [], []
    for out in tqdm(classifier(KeyDataset(hf_dataset, column_name), batch_size=BATCH_SIZE, truncation=True, max_length=512),
                    total=len(hf_dataset), desc=desc):
        emotions.append(out["label"])
        scores.append(round(out["score"] * 100, 2))
    return emotions, scores

# Load Data
print(f"Checking if INPUT_CSV exists: {os.path.exists(INPUT_CSV)}")
df = pd.read_csv(INPUT_CSV)
df["Prompt"] = df["Prompt"].fillna("")
df["Answer"] = df["Answer"].fillna("")
hf_dataset = Dataset.from_pandas(df)

# Setup Pipeline
classifier = pipeline("text-classification", model=MODEL_DIR, tokenizer=MODEL_DIR, device=0)

# # Score Prompts
prompt_emotions, prompt_scores = score_column(classifier, hf_dataset, "Prompt", "Scoring Prompts")
df["prompt_emotion"] = prompt_emotions
df["prompt_emotion_score"] = prompt_scores

# Score Answers
answer_emotions, answer_scores = score_column(classifier, hf_dataset, "Answer", "Scoring Answers")
df["answer_emotion"] = answer_emotions
df["answer_emotion_score"] = answer_scores

df.to_csv(OUTPUT_CSV, index=False)
print("Finished and saved to Google Drive! ✅")

Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

Scoring Answers:   0%|          | 0/112724 [00:00<?, ?it/s]

Finished and saved to Google Drive! ✅


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# Verify Google Drive mounted successfully
if os.path.ismount('/content/drive'):
    print("Google Drive mounted successfully.")
else:
    raise RuntimeError("Google Drive failed to mount.")

print(f"Listing contents")
!ls -F "/content/drive/MyDrive/devgpt-thesis/emotion-labelling"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully.
Listing contents
1_sources_metadata.csv			3_promptscored_dataset_output.csv
2_chatgpt_sharings.csv			codebert-se-emotion.zip
3_conversations_and_code_nlp_clean.csv	model/


### Additional Colab Environment Details for Thesis Report

Here are some code snippets to get more environment details for your thesis:


In [ ]:
# 1. Python Version
print("Python Version:")
!python --version

# 2. Key Library Versions
print("\nKey Library Versions:")
!pip freeze | grep -E 'transformers|datasets|pandas|tqdm|torch'

# 3. Detailed GPU Information (if using a GPU runtime)
print("\nDetailed GPU Information:")
!nvidia-smi

Python Version:
Python 3.12.13

Key Library Versions:
datasets==4.0.0
geopandas==1.1.4
pandas==2.2.2
pandas-datareader==0.11.1
pandas-gbq==0.30.0
pandas-stubs==2.2.2.240909
sentence-transformers==5.6.0
sklearn-pandas==2.2.0
tensorflow-datasets==4.9.10
torch==2.11.0+cu128
torchao==0.10.0
torchaudio==2.11.0+cu128
torchcodec==0.11.0+cu128
torchdata==0.11.0
torchsummary==1.5.1
torchtune==0.6.1
torchvision==0.26.0+cu128
tqdm==4.67.3
transformers==5.13.1
vega-datasets==0.9.0

Detailed GPU Information:
Wed Aug 12 19:12:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|      

You can include these outputs directly or

summarize them in your thesis under a 'Methodology' or 'Experimental Setup' section to provide a comprehensive overview of your development environment.